# 大規模データ・メモリ最適化

ダウンキャスト・チャンク読み込み・Polars/Dask で、大規模データを限られたメモリで扱う方法を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/advanced/06-big-data-memory`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# polars の確認・自動インストールと、未作成時のみチャンク処理・遅延評価用のサンプル CSV ("big.csv") を生成します
import os
import subprocess
import sys

try:
    import polars  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "polars"], check=True)

import numpy as np
import pandas as pd

if not os.path.exists("big.csv"):
    rng = np.random.default_rng(42)
    n_rows = 50_000
    sample_df = pd.DataFrame({
        "id": rng.choice(["A", "B", "C", "D"], size=n_rows),
        "y": rng.normal(100.0, 20.0, size=n_rows).round(2),
        "count": rng.integers(1, 500, size=n_rows),
    })
    sample_df.to_csv("big.csv", index=False)
    print("Created sample big.csv:", sample_df.shape)
else:
    sample_df = pd.read_csv("big.csv", nrows=50_000)
    print("Using existing big.csv (skipped sample generation):", sample_df.shape)

## メモリは有限

Kaggle Notebook のメモリには上限があり、大規模データは読み込むだけで枯渇します。
**型の最適化・分割読み込み・高速ライブラリ**でやりくりするのが上級者の必須スキルです。

## 型のダウンキャスト

pandas は既定で大きな型（int64/float64）を使います。値域に応じて小さい型へ落とすと、
メモリを半分以下にできることがあります。

In [ ]:
import numpy as np

def reduce_mem(df):
    for col in df.select_dtypes(include=["int", "float"]).columns:
        c_min, c_max = df[col].min(), df[col].max()
        if df[col].dtype.kind == "i":
            if c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype("int32")
        else:
            df[col] = df[col].astype("float32")
    return df

カテゴリ列は `astype("category")` でさらに削減できます。

## チャンク読み込み

一度に読めないファイルは、分割（チャンク）して処理します。

In [ ]:
chunks = pd.read_csv("big.csv", chunksize=1_000_000)
agg = sum(chunk.groupby("id")["y"].sum() for chunk in chunks)

## 高速ライブラリ

- **Polars**：Rust 製の高速・省メモリな DataFrame。遅延評価で大規模に強い
- **Dask**：pandas 風 API で分散/コア外処理。メモリに乗らないデータを扱える

In [ ]:
import polars as pl

df = pl.scan_csv("big.csv")                 # 遅延読み込み
out = df.group_by("id").agg(pl.col("y").mean()).collect()

In [ ]:
before_kb = sample_df.memory_usage(deep=True).sum() / 1024
reduced_df = reduce_mem(sample_df.copy())
after_kb = reduced_df.memory_usage(deep=True).sum() / 1024
print(f"Memory usage: {before_kb:.1f} KB -> {after_kb:.1f} KB")
print("Chunk sum:\n", agg)
print("Polars lazy mean:\n", out)

> **Tip:** まず `reduce_mem` と必要な列だけの読み込み（`usecols`）で乗り切れることも多いです。 それでも足りないときに Polars/Dask やチャンク処理へ進みましょう。

## 次のステップ

次は、チーム戦略（マージ・役割分担・提出管理）を学びます。